# 2. Eksplorasi Data (EDA)
Membaca `data_bersih.csv` dan melihat ukuran data, harga, sebaran, tren, dan inflasi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = '../data/processed/data_bersih.csv'
output_dir = '../data/processed'

KOL_PASAR = 'mkt_name'
KOL_PROVINSI = 'adm1_name'
KOL_TANGGAL = 'date'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
harga = [f'c_{k}' for k in KOMODITAS if f'c_{k}' in df.columns]
inflasi = [c for c in df.columns if c.startswith('inflation_')]
print(f'{len(df):,} baris | {df[KOL_PASAR].nunique()} pasar | {df[KOL_PROVINSI].nunique()} provinsi')
print(f'Periode: {df[KOL_TANGGAL].min():%b %Y} - {df[KOL_TANGGAL].max():%b %Y}')
print(f'Komoditas: {len(harga)} | kolom inflasi: {len(inflasi)}')
df.head()

## Statistik deskriptif harga penutupan (c_)

In [ ]:
df[harga].describe().T.round(0)

## Missing value pada harga dan inflasi

In [ ]:
kosong = df[harga + inflasi].isna().mean().mul(100).round(1).sort_values(ascending=False)
kosong.to_frame('persen_kosong')

In [ ]:
tahun_inflasi = df.groupby(df[KOL_TANGGAL].dt.year)[inflasi[0]].apply(lambda s: s.notna().mean() * 100).round(1) if inflasi else None
tahun_inflasi

## Jumlah pasar yang tercatat per tahun

In [ ]:
pasar_per_tahun = df.groupby(df[KOL_TANGGAL].dt.year)[KOL_PASAR].nunique()
pasar_per_tahun.plot(kind='bar', figsize=(10, 4), title='Jumlah pasar per tahun')
plt.ylabel('Jumlah pasar')
plt.show()

## Tren rata-rata harga bulanan

In [ ]:
df['bulan'] = df[KOL_TANGGAL].dt.to_period('M').dt.to_timestamp()
tren = df.groupby('bulan')[harga].mean()
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for ax, kol in zip(axes.ravel(), harga):
    tren[kol].plot(ax=ax, title=kol.replace('c_', ''))
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f'{output_dir}/grafik_tren_harga.png', dpi=150, bbox_inches='tight')
plt.show()

## Sebaran harga dan outlier (hanya dilihat, tidak diubah)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, kol in zip(axes.ravel(), harga):
    ax.boxplot(df[kol].dropna(), vert=False)
    ax.set_title(kol.replace('c_', ''))
plt.tight_layout()
plt.savefig(f'{output_dir}/grafik_boxplot_harga.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def jumlah_outlier(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    batas = 1.5 * (q3 - q1)
    return int(((s < q1 - batas) | (s > q3 + batas)).sum())

pd.Series({k: jumlah_outlier(df[k]) for k in harga}, name='jumlah_outlier_iqr').to_frame()

## Harga rata-rata beras per provinsi

In [ ]:
if 'c_rice' in df.columns:
    rata_prov = df.groupby(KOL_PROVINSI)['c_rice'].mean().sort_values()
    rata_prov.tail(15).plot(kind='barh', figsize=(8, 6), title='15 provinsi dengan harga beras tertinggi (IDR/kg)')
    plt.show()

## Inflasi tahunan

In [ ]:
for kol in inflasi[:1]:
    df.groupby('bulan')[kol].mean().plot(figsize=(12, 4), title=f'Rata-rata {kol}')
    plt.axhline(5, color='orange', linestyle='--')
    plt.axhline(10, color='red', linestyle='--')
    plt.ylabel('Inflasi (%)')
    plt.show()